# TPU Profiling

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()

In [ ]:
include("../src/WavKANConv.jl")

using .WavKANConv
using Lux, Lux: Training
using Reactant
using Optimisers
using Statistics
using Random
using MLDataDevices: reactant_device, cpu_device
using Printf

In [ ]:
dev = reactant_device()
rng = Lux.default_rng()
Random.seed!(42)
println("Device: ", dev)

## Data & Models

In [ ]:
MODEL_NAMES = ["CNN", "FNO", "KAN_CNN"]
CONFIGS = Dict(name => load_config(name) for name in MODEL_NAMES)

train_loader, test_loader = get_darcy_loader(1; dev=dev)
x_sample, y_sample = first(train_loader)

models = Dict{String, Any}()
for name in MODEL_NAMES
    cfg = CONFIGS[name]
    model = create_model(cfg)
    ps, st = Lux.setup(rng, model)
    models[name] = (; model, ps=ps|>dev, st=st|>dev, cfg, n_params=Lux.parameterlength(model))
    println("$name: $(models[name].n_params) params")
end

## Timing

In [ ]:
function benchmark(f, args...; n_warmup=3, n_trials=20)
    compile_t = @elapsed (compiled = Reactant.@compile f(args...))
    for _ in 1:n_warmup; compiled(args...); end
    times = [(@elapsed compiled(args...)) * 1000 for _ in 1:n_trials]
    return compiled, compile_t, times
end

## Forward Pass

In [ ]:
fwd_results = Dict{String, Any}()

@printf("%-20s %10s %10s %10s %10s\n", "Model", "Compile(s)", "Med(ms)", "Min(ms)", "Max(ms)")
println("-"^62)
for name in MODEL_NAMES
    m = models[name]
    fwd(x, ps, st) = m.model(x, ps, st)
    _, ct, t = benchmark(fwd, x_sample, m.ps, m.st)
    fwd_results[name] = Dict("compile_s"=>ct, "median_ms"=>median(t), "min_ms"=>minimum(t), "max_ms"=>maximum(t))
    @printf("%-20s %10.1f %10.3f %10.3f %10.3f\n", name, ct, median(t), minimum(t), maximum(t))
end

## Training Step (Forward + Backward + Param udpate)

In [ ]:
train_results = Dict{String, Any}()

@printf("%-20s %10s %10s %10s %10s\n", "Model", "Compile(s)", "Med(ms)", "Min(ms)", "Max(ms)")
println("-"^62)
for name in MODEL_NAMES
    m = models[name]
    loss_fn(y_pred, y) = loss_fcn(y_pred, y; p=m.cfg.p)
    objective(model, ps, st, (x, y)) = (loss_fn(first(model(x, ps, st)), y), last(model(x, ps, st)), (;))
    ts = Training.TrainState(m.model, m.ps, m.st, Optimisers.Adam(m.cfg.learning_rate))
    step(ts, x, y) = Training.single_train_step!(AutoEnzyme(), objective, (x, y), ts)

    _, ct, t = benchmark(step, ts, x_sample, y_sample; n_warmup=2, n_trials=15)
    train_results[name] = Dict("compile_s"=>ct, "median_ms"=>median(t), "min_ms"=>minimum(t), "max_ms"=>maximum(t))
    @printf("%-20s %10.1f %10.3f %10.3f %10.3f\n", name, ct, median(t), minimum(t), maximum(t))
end

## Batches

In [ ]:
BATCH_SIZES = [1, 5, 10, 25, 50]
scaling = Dict{String, Vector}()

for name in MODEL_NAMES
    m = models[name]
    scaling[name] = []
    println("\n$name:")
    @printf("  %5s %10s %12s\n", "BS", "Med(ms)", "Samples/s")
    for bs in BATCH_SIZES
        loader, _ = get_darcy_loader(bs; dev=dev)
        xb, _ = first(loader)
        fwd(x, ps, st) = m.model(x, ps, st)
        _, _, t = benchmark(fwd, xb, m.ps, m.st; n_warmup=2, n_trials=10)
        med = median(t)
        push!(scaling[name], (bs=bs, median_ms=med, throughput=bs/(med/1000)))
        @printf("  %5d %10.3f %12.1f\n", bs, med, bs/(med/1000))
    end
end

## Profiler Traces

Upload `.pb` files from `profiles/` to [Perfetto UI](https://ui.perfetto.dev).

In [ ]:
mkpath("profiles")
for name in MODEL_NAMES
    m = models[name]
    fwd(x, ps, st) = m.model(x, ps, st)
    compiled = Reactant.@compile fwd(x_sample, m.ps, m.st)
    compiled(x_sample, m.ps, m.st)

    tp = joinpath("profiles", name)
    mkpath(tp)
    Reactant.Profiler.with_profiler(tp) do
        for _ in 1:20; compiled(x_sample, m.ps, m.st); end
    end
    println("$name -> $tp")
end

## HLO

In [ ]:
const SYSTOLIC  = Set(["dot", "convolution"])
const ELEMWISE  = Set(["add","subtract","multiply","divide","maximum","minimum",
                       "exp","log","tanh","negate","abs","clamp","compare","select","convert"])
const REDUCTION = Set(["reduce","reduce-window"])
const LAYOUT    = Set(["reshape","transpose","broadcast","slice","concatenate",
                       "dynamic-slice","dynamic-update-slice","gather","scatter","pad"])

function analyze_hlo(hlo::String; mxu_align=128)
    counts = Dict(:systolic=>0, :elementwise=>0, :reduction=>0, :layout=>0, :other=>0)
    dots = 0; misaligned = 0
    for line in split(hlo, '\n')
        line = strip(line)
        startswith(line, '%') || continue
        m = match(r"%\S+\s*=\s*\S+\s+(\S+)\(", line)
        isnothing(m) && continue
        op = m.captures[1]
        cat = op in SYSTOLIC ? :systolic : op in ELEMWISE ? :elementwise :
              op in REDUCTION ? :reduction : op in LAYOUT ? :layout : :other
        counts[cat] += 1
        if op in SYSTOLIC
            dots += 1
            for sm in eachmatch(r"[fsbiu]\d+\[([^\]]+)\]", line)
                dims = parse.(Int, split(sm.captures[1], ','))
                if any(d -> d > 1 && d % mxu_align != 0, dims)
                    misaligned += 1; break
                end
            end
        end
    end
    return (; counts, total=sum(values(counts)), dots, misaligned)
end

@printf("%-20s %6s %8s %8s %8s %8s %8s %12s\n",
        "Model", "Total", "Systol", "Elem", "Reduce", "Layout", "Other", "Misaligned")
println("-"^88)
for name in MODEL_NAMES
    m = models[name]
    fwd(x, ps, st) = m.model(x, ps, st)
    hlo = try string(Reactant.@code_hlo fwd(x_sample, m.ps, m.st)) catch; "" end
    isempty(hlo) && continue
    r = analyze_hlo(hlo)
    @printf("%-20s %6d %8d %8d %8d %8d %8d %5d/%-5d\n",
            name, r.total, r.counts[:systolic], r.counts[:elementwise],
            r.counts[:reduction], r.counts[:layout], r.counts[:other],
            r.misaligned, r.dots)
end

## Print

In [ ]:
@printf("\n%-20s %8s %10s %10s %10s\n",
        "Model", "Params", "Fwd(ms)", "Train(ms)", "Compile(s)")
println("-"^60)
for name in MODEL_NAMES
    @printf("%-20s %8d %10.3f %10.3f %10.1f\n",
            name, models[name].n_params,
            fwd_results[name]["median_ms"],
            train_results[name]["median_ms"],
            train_results[name]["compile_s"])
end

## Save

In [ ]:
using JSON
open("benchmark_results.json", "w") do f
    write(f, JSON.json(Dict(
        "forward"=>fwd_results, "training"=>train_results, "scaling"=>scaling
    ), 2))
end
println("Saved to benchmark_results.json")